# Chapter 10 &mdash; Implementing the Matcher: `re2ast`, `dv`, `nullable`, `matches`

**Concept 7 of the Chapter 10 decomposition:** *Implementing the Matcher: `re2ast`, `dv`, `nullable`, and `matches`*

A tiny RE-to-AST compiler feeding a recursive matcher &mdash; about forty lines in total.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter10/Concept-Implementing-The-Matcher/Concept-Implementing-The-Matcher.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_rederiv    import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The whole implementation is four pieces:

* **`re2ast(s)`** &mdash; Jove's parser, returning `(ast, nodes, edges)`; the AST is
  nested tuples like `('.', (('str','0'), ('*', ('str','1'))))`;
* **`nullable(E)`** &mdash; the seven-case predicate;
* **`dv(c, E)`** &mdash; the nine-case derivative;
* **`matches(w, E)`** &mdash; the two-line driver.

The parser also accepts `!` for negation and `&` for intersection, which `re2nfa` does
not. `''` (or `""`) is $\varepsilon$; there is no surface syntax for $\emptyset$, so
the matcher introduces `PHI` itself.

Forty lines, and it handles operators the classical pipeline struggles with.

## 2. Definitions

### The matcher

In [ ]:
# --- the derivative matcher, in full -------------------------------------
# AST forms produced by re2ast:
#    ('@','@')            epsilon
#    ('str', c)           a single symbol
#    ('+', (E1, E2))      union
#    ('.', (E1, E2))      concatenation
#    ('*', E)             star
#    ('!', E)             negation
#    ('&', (E1, E2))      intersection
EPS   = ('@', '@')
PHI   = ('phi', 'phi')          # the empty language -- not produced by the
                                # parser, but the derivative rules need it

def nullable(E):
    t = E[0]
    if t == '@'  : return True
    if t == 'phi': return False
    if t == 'str': return False
    if t == '+'  : return nullable(E[1][0]) or  nullable(E[1][1])
    if t == '&'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '.'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '*'  : return True
    if t == '!'  : return not nullable(E[1])
    raise ValueError(E)

def dv(c, E):
    t = E[0]
    if t == '@'  : return PHI
    if t == 'phi': return PHI
    if t == 'str': return EPS if E[1] == c else PHI
    if t == '+'  : return ('+', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '&'  : return ('&', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '*'  : return ('.', (dv(c, E[1]), E))
    if t == '!'  : return ('!', dv(c, E[1]))
    if t == '.'  :
        E1, E2 = E[1]
        left = ('.', (dv(c, E1), E2))
        return ('+', (left, dv(c, E2))) if nullable(E1) else left
    raise ValueError(E)

def matches(s, E):
    for ch in s:
        E = dv(ch, E)
    return nullable(E)

def rmatch(restr, s):
    return matches(s, re2ast(restr)[0])

### Inspecting what the parser produces

In [ ]:
def show_ast(restr):
    ast, nodes, edges = re2ast(restr)
    print("RE   :", restr)
    print("AST  :", ast)
    print("nodes: %d, edges: %d  (these drive drawPT)" % (len(nodes), len(edges)))
    return ast

## 3. Tests

The parser's output, and the AST shapes.

In [ ]:
for r in ["''", "0", "0+1", "01", "0*", "!(0)", "0&1"]:
    print("%-8s -> %s" % (r, re2ast(r)[0]))

A nested example, with the node and edge lists the drawing uses.

In [ ]:
ast = show_ast("(0+1)*1")
assert ast[0] == '.'

The four pieces, composed.

In [ ]:
E = re2ast("(0+1)*01")[0]
print("nullable(E)        :", nullable(E))
print("dv('0', E) head    :", dv('0', E)[0])
print("matches('0101', E) :", matches('0101', E))
print("rmatch(...)        :", rmatch("(0+1)*01", "0101"))
assert rmatch("(0+1)*01", "0101") and not rmatch("(0+1)*01", "0110")

`!` and `&` are parsed here but **not** by `re2nfa`.

In [ ]:
import jove.Def_rederiv as RD, jove.Def_RE2NFA as RN
print("rederiv tokens :", RD.tokens)
print("re2nfa  tokens :", RN.tokens)
assert 'NOT' in RD.tokens and 'AND' in RD.tokens
assert 'NOT' not in RN.tokens and 'AND' not in RN.tokens
for r in ["!(0*)", "(0+1)&(1)"]:
    print("%-12s parsed by re2ast -> %s" % (r, re2ast(r)[0]))
print("\nre2nfa's lexer has no token for ! or & -- it reports 'Illegal character'.")

There is no surface syntax for $\emptyset$, so `PHI` is internal.

In [ ]:
print("PHI =", PHI, " nullable? ", nullable(PHI))
print("dv('0', PHI) =", dv('0', PHI))
assert not nullable(PHI) and dv('0', PHI) == PHI
print("\nPHI is absorbing: it can never become nullable again.")

A regression suite, against the classical route.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(10) for p in product('01', repeat=k)]
for r in ["0*1", "(0+1)*01", "(01)*", "0*1*", "1(0+1)*0", "(0+1)*1(0+1)(0+1)"]:
    D = min_dfa(nfa2dfa(re2nfa(r)))
    assert all(rmatch(r, s) == accepts_dfa(D, s) for s in strs)
    print("%-24s %d strings, no mismatch" % (r, len(strs)))

## 4. Exercises


1. Add a `size(E)` function and plot AST growth over a long input.
2. Implement $E^+$ in the parser *and* the matcher.
3. Where would you put memoization, and what would you key it on?

In [ ]:
# Your work for the exercises above.